In [16]:
import torch
import torch.nn as nn

In [ ]:
# Output size for a 2D transposed convolution layer
def check_transpose(I, K, S, P, OP=0):
    layer = nn.ConvTranspose2d(in_channels=1, out_channels=1, kernel_size=K,
                               stride=S, padding=P, output_padding=OP)
    x = torch.randn(1, 1, I, I)  # batch 1, channel 1, I x I feature map
    return tuple(layer(x).shape[-2:])

# Question 1

In [21]:
actual = check_transpose(I=8, K=4, S=2, P=1, OP=1)
print(f"PyTorch output: {actual}\n")

PyTorch output: (17, 17)



# Qestion 2

In [20]:
for S in (2, 3):
    actual = check_transpose(I=8, K=4, S=S, P=1, OP=1)
    print(f"  stride {S}:  PyTorch output: {actual}")
print()

  stride 2:  PyTorch output: (17, 17)
  stride 3:  PyTorch output: (24, 24)



# Question 3

$$
O = (I - 1) \times S - 2P + K + OP
$$

# Question 4

In [18]:
for K in range(1, 33):
    for S in range(1, 33):
        if conv_transpose_output_size(I=16, K=K, S=S, P=0) == 32:
            _, actual = check_transpose(I=16, K=K, S=S, P=0)
            print(f"  kernel {K}, stride {S}  ->  PyTorch output: {actual}")
print()

  kernel 2, stride 2  ->  PyTorch output: 32
  kernel 17, stride 1  ->  PyTorch output: 32



# Question 5

In [17]:
x = torch.tensor([6.0, 8.0, 10.0, 6.0])
bn = nn.BatchNorm1d(num_features=1, affine=False)
bn.train()                                         
out = bn(x.view(4, 1))                            

print("BatchNorm:", [round(v, 4) for v in out.view(-1).tolist()])

BatchNorm: [-0.9045, 0.3015, 1.5076, -0.9045]


# Question 6

### ReLU vs LeakyReLU

**ReLU:**
$$
\text{ReLU}(x) = \max(0, x) =
\begin{cases}
x, & x > 0 \\
0, & x \le 0
\end{cases}
$$

**LeakyReLU:**
$$
\text{LeakyReLU}(x) =
\begin{cases}
x, & x > 0 \\
\alpha x, & x \le 0
\end{cases}
$$

$\alpha$ is a small slope.

The key difference: for $x < 0$, ReLU's slope is $0$, while LeakyReLU's slope is $\alpha$.

# Question 7

The network learns through the gradients, and a gradient of zero means no learning with the model. ReLU can produce zero based on the formula, and LeakyReLU can't.
- When x input is negative, the slope for ReLU is 0 and that for LeakyReLU is $\alpha$.
- In a GAN, the generator never sees real data or a correct answer. Its only feedback is the gradient that travels backward through the discriminator. When the input for the discriminator is identified as "fake", using ReLU, the gradient sent backforward to the generator is 0. So, the generator actually learns nothing from a gradient of 0. Instead of this,using LeakyReLU, when the discriminator confidently rejects a fake, the generator still gets a useful direction to improve.